![image](https://raw.githubusercontent.com/IBM/watsonx-ai-samples/master/cloud/notebooks/headers/watsonx-Prompt_Lab-Notebook.png)

# Use lm-evaluation-harness and own benchmarking data with watsonx.ai foundation models

This notebook contains the steps and code to demonstrate usage of `lm-evaluation-harness` (also called `lm-eval`) package with `ibm_watsonx_ai` and `watsonx_llm` language model.   

Some familiarity with Python is helpful. This notebook uses Python 3.12.

## Learning goals

The learning goals of this notebook are: 
1. Setting up `lm-evaluation-harness` and `ibm_watsonx_ai`
2. Basic `lm-evaluation-harness` usage with available tasks
3. Preparing custom tasks and setting up local datasets
4. Calling `lm-evaluation-harness` with locally prepared tasks

## Table of contents
1. [Set up the environment](#Set-up-the-environment)
2. [Basic lm-evaluation-harness usage](#Basic-lm-evaluation-harness-usage)
3. [Prepare own data for benchmarking](#Prepare-own-data-for-benchmarking)
4. [Run benchmarks with local data](#Run-benchmarks-with-local-data)
5. [Summary and next steps](#Summary-and-next-steps)

<a id="Set-up-the-environment"></a>

## Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

- Create a <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance (a free plan is offered and information about how to create the instance can be found <a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/wml-plans.html?context=wx&audience=wdp" target="_blank" rel="noopener no referrer">here</a>).
- Create a [Cloud Object Storage (COS) instance](https://console.bluemix.net/catalog/infrastructure/cloud-object-storage) (a lite plan is offered and information about how to order storage can be found [here](https://cloud.ibm.com/docs/cloud-object-storage/basics/order-storage.html#order-storage)).
  
__Note: When using Watson Studio, you already have a COS instance associated with the project you are running the notebook in.__

### Install dependencies <a name="install-packages"></a>
**Note:** `ibm-watsonx-ai` documentation can be found <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">here</a>.

**Note:** `lm-evaluation-harness` documentation can be found <a href="https://github.com/EleutherAI/lm-evaluation-harness/tree/main/docs" target="_blank" rel="noopener no referrer">here</a>.

In [1]:
%pip install wget | tail -n 1
%pip install pyyaml | tail -n 1
%pip install lm-eval[ibm_watsonx_ai] | tail -n 1

### Connection to watsonx.ai Runtime

Authenticate the watsonx.ai Runtime service on IBM Cloud. You need to provide Cloud `API key` and `location`.

**Tip**: Your `Cloud API key` can be generated by going to the [**Users** section of the Cloud console](https://cloud.ibm.com/iam#/users). From that page, click your name, scroll down to the **API Keys** section, and click **Create an IBM Cloud API key**. Give your key a name and click **Create**, then copy the created key and paste it below. You can also get a service specific url by going to the [**Endpoint URLs** section of the watsonx.ai Runtime docs](https://cloud.ibm.com/apidocs/machine-learning).  You can check your instance location in your  <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance details.


You can use [IBM Cloud CLI](https://cloud.ibm.com/docs/cli/index.html) to retrieve the instance `location`.

```
ibmcloud login --apikey API_KEY -a https://cloud.ibm.com
ibmcloud resource service-instance INSTANCE_NAME
```


**NOTE:** You can also get a service specific apikey by going to the [**Service IDs** section of the Cloud Console](https://cloud.ibm.com/iam/serviceids).  From that page, click **Create**, and then copy the created key and paste it in the following cell.  


**Action**: Enter your `api_key` and `location` in the following cell.

In [2]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    url="https://us-south.ml.cloud.ibm.com",
    api_key=getpass.getpass("Enter your watsonx.ai api key and hit enter: "),
)

<a id="Working-with-projects"></a>
### Working with projects

You need to create a project that will be used for your work. If you do not have a space, you can use [Projects Dashboard](https://dataplatform.cloud.ibm.com/wx/home?context=wx) to create one.

- Click __Create a new project__
- Provide a name
- Select Cloud Object Storage
- Select Watson Machine Learning instance and press __Create__
- Copy `project_id` and paste it below

__Action__: Assign project ID below

In [3]:
import os

try:
    project_id = os.environ["PROJECT_ID"]
except KeyError:
    project_id = input("Enter your project_id and hit enter: ")

<a id="Export-environment-variables-to-be-used-by-lm-evaluation-harness"></a>
### Export environment variables to be used by lm-evaluation-harness

In [4]:
os.environ["WATSONX_API_KEY"] = str(credentials.api_key)
os.environ["WATSONX_URL"] = str(credentials.url)
os.environ["WATSONX_PROJECT_ID"] = str(project_id)

### Find lm-evaluation-harness executable path

In [5]:
import sys
from pathlib import Path

lm_eval_path = str(Path(sys.executable).parent / "lm-eval")

<a id="Basic-lm-evaluation-harness-usage"></a>
## Basic lm-evaluation-harness usage

In this section, we will perform a sample call using the `watsonx_llm` and an available `gsm8k` task.

**Note:** You can also execute this command using CLI:
```sh
lm-eval --model watsonx_llm \
        --verbosity ERROR \
        --model_args '{"model_id": "ibm/granite-4-h-small", "generate_params": {"max_new_tokens": 256}}' \
        --limit 10 \
        --tasks gsm8k
```

In [6]:
import json
import subprocess

model_args = {
    "model_id": "ibm/granite-4-h-small",
    "generate_params": {"max_new_tokens": 256},
}

arguments = {
    "model": "watsonx_llm",
    "verbosity": "ERROR",
    "model_args": json.dumps(model_args),
    "limit": "10",
    "tasks": "gsm8k",
}

command = [lm_eval_path]
for key, value in arguments.items():
    command.append(f"--{key}")
    command.append(value)

subprocess.run(command, check=True)

2026-05-18:16:16:39 WARNING  [config.evaluate_config:287] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-05-18:16:16:54 INFO     [_cli.run:388] Selected Tasks: ['gsm8k']
2026-05-18:16:16:54 INFO     [evaluator:162] Setting verbosity through simple_evaluate is deprecated.
2026-05-18:16:16:54 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-05-18:16:16:54 INFO     [evaluator:239] Initializing watsonx_llm model, with arguments: {'model_id': 'ibm/granite-4-h-small', 'generate_params': {'max_new_tokens': 256}}
2026-05-18:16:16:59 INFO     [evaluator_utils:446] Selected tasks:
2026-05-18:16:16:59 INFO     [evaluator_utils:480] Task: gsm8k (gsm8k/gsm8k.yaml)
2026-05-18:16:16:59 INFO     [evaluator:314] gsm8k: Using gen_kwargs: {'until': ['Question:', '</s>', '<|im_end|>'], 'do_sample': False, 'temperature': 0.0}
2026-05-18:16:16:59 INFO  

watsonx_llm ({'model_id': 'ibm/granite-4-h-small', 'generate_params': {'max_new_tokens': 256}}), gen_kwargs: ({}), limit: 10.0, num_fewshot: None, batch_size: 1
|Tasks|Version|     Filter     |n-shot|  Metric   |   |Value|   |Stderr|
|-----|------:|----------------|-----:|-----------|---|----:|---|-----:|
|gsm8k|      3|flexible-extract|     5|exact_match|↑  |  0.9|±  |   0.1|
|     |       |strict-match    |     5|exact_match|↑  |  0.9|±  |   0.1|



CompletedProcess(args=['lm-eval', '--model', 'watsonx_llm', '--verbosity', 'ERROR', '--model_args', '{"model_id": "ibm/granite-4-h-small", "generate_params": {"max_new_tokens": 256}}', '--limit', '10', '--tasks', 'gsm8k'], returncode=0)

If you get a following error: `RuntimeError: Model [model_id] is not supported: does not return logprobs for input tokens` try again with a different model that has the `logprobs` enabled. Available models can be found [here](https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/fm-models.html?context=wx).

<a id="Prepare-own-data-for-benchmarking"></a>
## Prepare own data for benchmarking

<a id="Prepare-an-APIClient-instance"></a>
### Prepare an APIClient instance

In [7]:
from ibm_watsonx_ai import APIClient

api_client = APIClient(credentials=credentials, project_id=project_id)

### Prepare data assets <a name="prepare-assets"></a>

This example uses the `date_understanding/test-00000-of-00001.parquet` datasets from [SaylorTwift/bbh](https://huggingface.co/datasets/SaylorTwift/bbh) dataset. If you wish to use your own validation data asset, provide its asset ID below.

In [8]:
import wget

validation_filename = "test-00000-of-00001.parquet"

validation_asset_id = input(
    "Enter your validation data asset ID. If you wish to use the default data asset, hit enter: "
)

if validation_asset_id:
    # Download existing data asset
    path = Path(os.getcwd()) / validation_filename

    if not path.exists():
        api_client.data_assets.download(
            asset_id=validation_asset_id, filename=validation_filename
        )
else:
    # Create default data asset
    base_url = "https://huggingface.co/datasets/SaylorTwift/bbh/resolve/main/date_understanding/"

    path = Path(os.getcwd()) / validation_filename

    if not path.exists():
        wget.download(f"{base_url}{validation_filename}")

    asset_details = api_client.data_assets.create(
        file_path=validation_filename, name=validation_filename
    )

    validation_asset_id = api_client.data_assets.get_id(asset_details)

Creating data asset...
SUCCESS


<a id="Sample-task-definition"></a>
### Sample task definition
For this section we will be using the [`bbh_cot_fewshot_date_understanding`](https://github.com/EleutherAI/lm-evaluation-harness/blob/main/lm_eval/tasks/bbh/cot_fewshot/date_understanding.yaml) task as an example on how to build a task and execute it from outside of the `lm-evaluation-harness` repository. Tasks for benchmarking are stored as `yaml` files. You can take a look at the [data understanding dataset](https://raw.githubusercontent.com/suzgunmirac/BIG-Bench-Hard/refs/heads/main/bbh/date_understanding.json) and its [corresponding task](https://github.com/EleutherAI/lm-evaluation-harness/blob/main/lm_eval/tasks/bbh/cot_fewshot/date_understanding.yaml).

Normally, the `dataset_path` and `dataset_name` point to datasets that are stored in `HuggingFace` hub and the `task` points to the list of tasks registered inside the `lm-evaluation-harness` repo. However, it's possible to point to a local dataset with a custom made task. In order to do so, the user needs to specify the local paths in the `dataset_kwargs` field and the files type in the `dataset_path` field:

```yaml
dataset_path: file_type (arrow, parquet, jsonl...)
dataset_kwargs:
  data_files:
    train: /path/to/train/train_file
    validation: /path/to/validation/validation_file
    test: /path/to/test/test_file
```
It is also necessary to have the local `yaml` file saved to a specific path and this path needs to be included when calling the `lm-eval` command. Knowing what should be included in the task structure we can recreate a dictionary with this info. 

In [9]:
task = {
    "dataset_path": "parquet",
    "dataset_kwargs": {
        "data_files": {"validation": str(path)},
    },
    "output_type": "generate_until",
    "doc_to_target": "{{target}}",
    "target_delimiter": "",
    "validation_split": "validation",
    "metric_list": [
        {"metric": "exact_match", "aggregation": "mean", "higher_is_better": True}
    ],
    "generation_kwargs": {
        "max_gen_toks": 1024,
        "until": ["</s>", "Q", "\n\n"],
        "do_sample": False,
        "temperature": 0,
    },
    "filter_list": [
        {
            "name": "get-answer",
            "filter": [
                {"function": "regex", "regex_pattern": "(?<=the answer is )(.*)(?=.)"},
                {"function": "take_first"},
            ],
        }
    ],
    "num_fewshot": 3,
    "metadata": {"version": 4},
    "description": "Infer the date from context.\n\n",
    "doc_to_text": "Q: {{input}}\nA: Let's think step by step.\n",
    "fewshot_config": {
        "sampler": "first_n",
        "samples": [
            {
                "input": "Today is Christmas Eve of 1937. What is the date 10 days ago in MM/DD/YYYY?\nOptions:\n(A) 12/14/2026\n(B) 12/14/1950\n(C) 12/14/2007\n(D) 12/14/1937\n(E) 07/14/1938\n(F) 12/14/1988",
                "target": "If today is Christmas Eve of 1937, then today's date is December 24, 1937. 10 days before today is December 14, 1937, that is 12/14/1937. So the answer is (D).",
            },
            {
                "input": "Tomorrow is 11/12/2019. What is the date one year ago from today in MM/DD/YYYY?\nOptions:\n(A) 09/04/2018\n(B) 11/11/2018\n(C) 08/25/2018\n(D) 11/02/2018\n(E) 11/04/2018",
                "target": "If tomorrow is 11/12/2019, then today is 11/11/2019. The date one year ago from today is 11/11/2018. So the answer is (B).",
            },
            {
                "input": "Jane and John married on Jan 2, 1958. It is their 5-year anniversary today. What is the date tomorrow in MM/DD/YYYY?\nOptions:\n(A) 01/11/1961\n(B) 01/03/1963\n(C) 01/18/1961\n(D) 10/14/1960\n(E) 01/03/1982\n(F) 12/03/1960",
                "target": "If Jane and John married on Jan 2, 1958, then and if it is their 5-year anniversary today, then today's date is Jan 2, 1963. The date tomorrow is Jan 3, 1963, that is 01/03/1963. So the answer is (B).",
            },
        ],
    },
    "task": "test_task_local",
}

Save custom task to a YAML file

In [10]:
import yaml

with open("test_task.yaml", "w", encoding="utf-8") as yaml_file:
    yaml.dump(task, yaml_file, default_flow_style=False)

<a id="Run-benchmarks-with-local-data"></a>
## Run benchmarks with local data

Having the datasets and the yaml task stored, we can run the lm-eval command with the `--include_path .` argument that will point to the local path and the local task name (`test_task_local`). Evaluation results will be saved to the specified (`results`) directory. 

```sh
lm_eval --model watsonx_llm \
        --model_args '{"model_id": "ibm/granite-4-h-small", "generate_params": {"max_new_tokens": 256}}' \
        --include_path . \
        --limit 10 \
        --tasks test_task_local \
        --output_path results

In [11]:
model_args = {
    "model_id": "ibm/granite-4-h-small",
    "generate_params": {"max_new_tokens": 256},
}

arguments = {
    "model": "watsonx_llm",
    "model_args": json.dumps(model_args),
    "include_path": ".",
    "limit": "10",
    "tasks": "test_task_local",
    "output_path": "results",
}

command = [lm_eval_path]
for key, value in arguments.items():
    command.append(f"--{key}")
    command.append(value)

subprocess.run(command, check=True)

2026-05-18:16:17:37 WARNING  [config.evaluate_config:287] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-05-18:16:17:41 INFO     [_cli.run:387] Including path: .
2026-05-18:16:17:41 INFO     [_cli.run:388] Selected Tasks: ['test_task_local']
2026-05-18:16:17:41 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-05-18:16:17:41 INFO     [evaluator:239] Initializing watsonx_llm model, with arguments: {'model_id': 'ibm/granite-4-h-small', 'generate_params': {'max_new_tokens': 256}}
Generating validation split: 250 examples [00:00, 99674.52 examples/s]
2026-05-18:16:17:43 INFO     [evaluator_utils:446] Selected tasks:
2026-05-18:16:17:43 INFO     [evaluator_utils:480] Task: test_task_local (test_task.yaml)
2026-05-18:16:17:43 INFO     [evaluator:314] test_task_local: Using gen_kwargs: {'do_sample': False, 'max_gen_toks': 1024, 'temperatur

watsonx_llm ({'model_id': 'ibm/granite-4-h-small', 'generate_params': {'max_new_tokens': 256}}), gen_kwargs: ({}), limit: 10.0, num_fewshot: None, batch_size: 1
|     Tasks     |Version|  Filter  |n-shot|  Metric   |   |Value|   |Stderr|
|---------------|------:|----------|-----:|-----------|---|----:|---|-----:|
|test_task_local|      4|get-answer|     3|exact_match|↑  |  0.8|±  |0.1333|



CompletedProcess(args=['lm-eval', '--model', 'watsonx_llm', '--model_args', '{"model_id": "ibm/granite-4-h-small", "generate_params": {"max_new_tokens": 256}}', '--include_path', '.', '--limit', '10', '--tasks', 'test_task_local', '--output_path', 'results'], returncode=0)

Now let's see the evaluation results. The file name consists of the `results` prefix and a unique timestamp. 

**Note:** For pretty printing reasons, the `pretty_env_info` field is excluded from output.

In [12]:
results_dir = Path.cwd() / "results"
results_files_dir = next(filter(lambda x: x.is_dir(), results_dir.iterdir()))
file_paths = sorted(results_files_dir.iterdir(), key=lambda x: x.stat().st_ctime)

data: dict = json.loads(file_paths[0].read_bytes())
data.pop("pretty_env_info", None)

print(json.dumps(data, indent=2))

{
  "results": {
    "test_task_local": {
      "name": "test_task_local",
      "alias": "test_task_local",
      "sample_len": 10,
      "exact_match,get-answer": 0.8,
      "exact_match_stderr,get-answer": 0.13333333333333333
    }
  },
  "group_subtasks": {},
  "configs": {
    "test_task_local": {
      "task": "test_task_local",
      "dataset_path": "parquet",
      "dataset_kwargs": {
        "data_files": {
          "validation": "test-00000-of-00001.parquet"
        }
      },
      "validation_split": "validation",
      "doc_to_text": "Q: {{input}}\nA: Let's think step by step.\n",
      "doc_to_target": "{{target}}",
      "unsafe_code": false,
      "description": "Infer the date from context.\n\n",
      "target_delimiter": "",
      "fewshot_delimiter": "\n\n",
      "fewshot_config": {
        "sampler": "first_n",
        "split": null,
        "process_docs": null,
        "fewshot_indices": null,
        "samples": [
          {
            "input": "Today is Chris

<a id="Summary-and-next-steps"></a>
## Summary and next steps
You successfully completed this notebook!

You learned how to use `ibm-watsonx-ai` and `lm-evaluation-harness` to run custom local and registered benchmarks.

Check out our [Online Documentation](https://www.ibm.com/cloud/watson-studio/autoai) for more samples, tutorials, documentation, how-tos, and blog posts.

### Authors 

**Marta Tomzik**, Software Engineer at Watson Machine Learning.

**Rafał Chrzanowski**, Software Engineer at watsonx.ai.

Copyright © 2025-2026 IBM. This notebook and its source code are released under the terms of the MIT License.